# Download gridded radar data from thredds

In [1]:
import pandas as pd
import datetime
import numpy as np
import xarray as xr
import netCDF4
import matplotlib.pyplot as plt
import tqdm

### challenges: 
- takes alot of time, so the script must be able to restart from crash
    - store latest successfully stored datetime
    - create netcdf first, save iteratively
- takes alot of data, allocate before running the loop
    - store as int in multiplies of 100

In [2]:
start = '2022-06-25'
end = '2022-07-01'
timesteps = pd.date_range(start=start, end=end, freq="1h")


# Download daily raw files

In [5]:
# download metadata
link = 'https://thredds.met.no/thredds/dodsC/metpparchivev3/2022/06/25/met_analysis_1_0km_nordic_20220625T23Z.nc'
ds = netCDF4.Dataset(link)

In [3]:
# download metadata
link = 'https://thredds.met.no/thredds/dodsC/metpparchivev3/2022/06/25/met_analysis_1_0km_nordic_20220625T23Z.nc'
ds = netCDF4.Dataset(link)
lat_norway = ds['latitude'][:].data
lon_norway = ds['longitude'][:].data

# save
np.save('/media/erlend/a7db2311-330d-408b-b1d2-57343136083f/model/lat_norway', lat_norway)
np.save('/media/erlend/a7db2311-330d-408b-b1d2-57343136083f/model/lon_norway', lon_norway)
ds.close()

In [4]:
# Download hour by hour
time_recorded = []

for time in tqdm.tqdm(timesteps): 
    hour = time.strftime('%H')
    day = time.strftime('%d')
    month = time.strftime('%m')
    year = time.strftime('%Y')
    
    # setup link to threds: 
    try:
        #link = 'https://thredds.met.no/thredds/dodsC/metppltcarchivev1/2022/06/25/met_analysis_ltc_1_0km_nordic_20220625T23Z.nc'

        link = 'https://thredds.met.no/thredds/dodsC/metpparchivev3/'+ year + '/' + month + '/' + day + '/met_analysis_1_0km_nordic_'+ year + month + day + 'T'+ hour +'Z.nc'
        ds = netCDF4.Dataset(link)

        # get times for this day
        times = ds['time'][:].data

        # download precipitation ammounts for all gridcells
        precipitation = ds.variables['precipitation_amount'][:, :, :].data.astype('float16') # converts to less precise, but less memory.
        np.save('/media/erlend/a7db2311-330d-408b-b1d2-57343136083f/model/prec_' +str(year)+str(month)+str(day)+str(hour), precipitation)
        np.save('/media/erlend/a7db2311-330d-408b-b1d2-57343136083f/model/times_' +str(year)+str(month)+str(day)+str(hour), times)

    finally:
        #clean up
        ds.close()
        del precipitation
        del times
        del ds

        # record those that were stored
        time_recorded.append(time) # in case we miss something, go back

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 145/145 [03:14<00:00,  1.34s/it]


# Read read raw files and concat to netcdf

In [5]:
import pandas as pd
import datetime
import numpy as np
import xarray as xr
import netCDF4
import matplotlib.pyplot as plt
import tqdm

In [6]:
lat_norway = np.load('/media/erlend/a7db2311-330d-408b-b1d2-57343136083f/model/lat_norway.npy')
lon_norway = np.load('/media/erlend/a7db2311-330d-408b-b1d2-57343136083f/model/lon_norway.npy')


In [7]:
# Create a new netCDF file
dataset = netCDF4.Dataset('/media/erlend/a7db2311-330d-408b-b1d2-57343136083f/model/weather_data.nc', 'w', format='NETCDF4_CLASSIC')

# Define the dimensions
time_dim = dataset.createDimension('time', None)  # the unlimited dimension (None, can be appended to)
y_dim = dataset.createDimension('y', 2321)
x_dim = dataset.createDimension('x', 1796)

# Define the variables
times = dataset.createVariable('time', np.float64, ('time',))
times.units = 'minutes since 2022-01-01 00:00:00'
times.calendar = 'gregorian'

ys = dataset.createVariable('y', np.int32, ('y',))
xs = dataset.createVariable('x', np.int32, ('x',))
lats = dataset.createVariable('latitude', np.float32, ('y', 'x'))  # latitudes
lons = dataset.createVariable('longitude', np.float32, ('y', 'x'))  # longitudes
values = dataset.createVariable('value', np.int16, ('time', 'y', 'x'))  # the data variable

# Set the variable values
xs[:] = np.arange(1796)
ys[:] = np.arange(2321)

lats[:, :] = lat_norway
lons[:, :] = lon_norway


In [8]:
# For through days:
i = 0
for time in tqdm.tqdm(timesteps): 
    hour = time.strftime('%H')
    day = time.strftime('%d')
    month = time.strftime('%m')
    year = time.strftime('%Y')
    
    # get times for this day
    times_read = np.load('/media/erlend/a7db2311-330d-408b-b1d2-57343136083f/model/times_' +str(year)+str(month)+str(day)+str(hour)+'.npy')
    precipitation = (np.load('/media/erlend/a7db2311-330d-408b-b1d2-57343136083f/model/prec_' +str(year)+str(month)+str(day)+str(hour)+'.npy')*100).astype('int16')
    isnan = np.isnan(precipitation)
    precipitation[isnan] == -1
    times_read = np.array([datetime.datetime.utcfromtimestamp(i) for i in times_read])

    # write to NETCDF in chunks (speeds up later reading)
    times[i:i+len(times_read)] = netCDF4.date2num(times_read, units=times.units , calendar=times.calendar)
    values[i:i+len(times_read), :, :] = precipitation
    i += len(times_read)

# Close the file
dataset.close()

  0%|                                                                                                                     | 0/145 [00:00<?, ?it/s]/tmp/ipykernel_220073/2136447976.py:14: DeprecationWarning: datetime.datetime.utcfromtimestamp() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.fromtimestamp(timestamp, datetime.UTC).
  times_read = np.array([datetime.datetime.utcfromtimestamp(i) for i in times_read])
100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 145/145 [00:06<00:00, 21.87it/s]
